In [1]:
# Data processing: merge Delhi AQI with Punjab fire and wind daily aggregates (with distance-weighted FRP & wind-direction)
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path(r"a:/Software Projects/Delhi-AQI-Model/data")
OUT_CSV = DATA_DIR / "merged_aqi_fire_wind.csv"

# Helper geospatial functions
from math import radians, cos, sin, asin, sqrt, atan2, degrees

def haversine_km(lat1, lon1, lat2, lon2):
    # return distance in kilometers
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return 6371.0 * 2 * asin(sqrt(a))

def bearing_deg(lat1, lon1, lat2, lon2):
    # bearing from point1 (lat1,lon1) to point2 in degrees (0-360)
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = sin(dlon) * cos(lat2)
    y = cos(lat1)*sin(lat2) - sin(lat1)*cos(lat2)*cos(dlon)
    br = (degrees(atan2(x, y)) + 360) % 360
    return br

# Delhi reference point (approx centroid)
delhi_lat, delhi_lon = 28.7041, 77.1025

# 1) Load Delhi AQI and build a proper date column
aqi = pd.read_csv(DATA_DIR / 'delhi_aqi_new.csv')
# columns: Date (day), Month, Year
aqi = aqi.rename(columns={'Date':'day'})
aqi['date'] = pd.to_datetime(aqi[['Year','Month','day']])
aqi = aqi.sort_values('date')
aqi_daily = aqi[['date','AQI','PM2.5','PM10','NO2','SO2','CO','Ozone']].copy()

# 1b) compute daily mean wind direction for Delhi from higher-res station data (circular mean)
delhi_wind_file = DATA_DIR / 'delhi_air_quality_2024.csv'
try:
    ddf = pd.read_csv(delhi_wind_file, parse_dates=['event_timestamp'], usecols=['event_timestamp','wind_direction'])
    ddf = ddf.dropna(subset=['wind_direction'])
    ddf['date'] = ddf['event_timestamp'].dt.normalize()
    ddf['wind_direction'] = pd.to_numeric(ddf['wind_direction'], errors='coerce')
    # vector mean
    ddf['wd_rad'] = np.deg2rad(ddf['wind_direction'])
    grp = ddf.groupby('date').agg(wd_cos_mean=('wd_rad', lambda s: np.cos(s).mean()), wd_sin_mean=('wd_rad', lambda s: np.sin(s).mean()))
    grp = grp.reset_index()
    grp['mean_wind_dir_rad'] = np.arctan2(grp['wd_sin_mean'], grp['wd_cos_mean'])
    grp['mean_wind_dir_deg'] = (np.rad2deg(grp['mean_wind_dir_rad']) + 360) % 360
    delhi_daily_wind_dir = grp[['date','mean_wind_dir_deg','wd_cos_mean','wd_sin_mean']].rename(columns={'wd_cos_mean':'mean_wind_dir_cos','wd_sin_mean':'mean_wind_dir_sin'})
except Exception as e:
    print('Could not compute Delhi daily wind direction:', e)
    delhi_daily_wind_dir = pd.DataFrame(columns=['date','mean_wind_dir_deg','mean_wind_dir_cos','mean_wind_dir_sin'])

# 2) Aggregate fires by date (use chunked read for large files and filter to Punjab bounding box)
fire_files = [DATA_DIR / 'punjab_fires_2024.csv', DATA_DIR / 'punjab_fire_2020-2024.csv', DATA_DIR / 'fire_archive_J1V-C2_696273.csv']
# Punjab rough bounding box
lat_min, lat_max = 28.0, 33.5
lon_min, lon_max = 73.0, 77.5

def aggregate_fire_chunks(file_path, date_col='acq_date', lat_col='latitude', lon_col='longitude', frp_col='frp'):
    chunks = pd.read_csv(file_path, parse_dates=[date_col], chunksize=200_000)
    agg_list = []
    for c in chunks:
        c = c[(c[lat_col].between(lat_min, lat_max)) & (c[lon_col].between(lon_min, lon_max))]
        if c.empty:
            continue
        c[date_col] = pd.to_datetime(c[date_col]).dt.normalize()
        # compute distance to Delhi and bearing
        c['dist_km'] = c.apply(lambda r: haversine_km(r[lat_col], r[lon_col], delhi_lat, delhi_lon), axis=1)
        c['bearing_to_delhi'] = c.apply(lambda r: bearing_deg(r[lat_col], r[lon_col], delhi_lat, delhi_lon), axis=1)
        # distance weight (simple inverse distance with +1 km to avoid div-by-zero)
        c['frp_weighted'] = c[frp_col] / (c['dist_km'] + 1.0)
        # bearing vector weighted by FRP (for circular mean)
        c['b_x'] = np.cos(np.deg2rad(c['bearing_to_delhi'])) * c[frp_col]
        c['b_y'] = np.sin(np.deg2rad(c['bearing_to_delhi'])) * c[frp_col]
        # aggregate
        g = c.groupby(date_col).agg(
            fire_count=(frp_col,'count'),
            total_frp=(frp_col,'sum'),
            frp_weighted_sum=('frp_weighted','sum'),
            dist_sum=('dist_km','sum'),
            min_dist_km=('dist_km','min'),
            b_x_sum=('b_x','sum'),
            b_y_sum=('b_y','sum'),
            max_frp=(frp_col,'max')
        )
        agg_list.append(g)
    if not agg_list:
        return pd.DataFrame()
    df = pd.concat(agg_list).groupby(level=0).sum()
    df = df.reset_index().rename(columns={date_col:'date'})
    # compute means
    df['mean_frp'] = df['total_frp'] / df['fire_count']
    df['mean_dist_km'] = df['dist_sum'] / df['fire_count']
    # circular mean bearing (weighted by FRP)
    df['mean_bearing_deg'] = (np.rad2deg(np.arctan2(df['b_y_sum'], df['b_x_sum'])) + 360) % 360
    # rename weighted FRP column
    df = df.rename(columns={'frp_weighted_sum':'weighted_frp'})
    df = df[['date','fire_count','total_frp','mean_frp','max_frp','weighted_frp','mean_dist_km','min_dist_km','mean_bearing_deg']]
    return df

# Process fires - try smaller file first to speed up
fire_daily = aggregate_fire_chunks(DATA_DIR / 'punjab_fires_2024.csv')
# Try adding larger archives (wrap in try to avoid memory issues)
for f in [DATA_DIR / 'punjab_fire_2020-2024.csv', DATA_DIR / 'fire_archive_J1V-C2_696273.csv']:
    try:
        more = aggregate_fire_chunks(f)
        if not more.empty:
            combined = pd.concat([fire_daily.set_index('date'), more.set_index('date')], axis=0)
            # sum sensible aggregates
            fire_daily = combined.groupby(combined.index).agg({
                'fire_count':'sum', 'total_frp':'sum', 'weighted_frp':'sum', 'mean_frp':'sum', 'max_frp':'max', 'mean_dist_km':'sum', 'min_dist_km':'min', 'mean_bearing_deg':'mean'
            }).reset_index()
            fire_daily['mean_frp'] = fire_daily['total_frp'] / fire_daily['fire_count']
            fire_daily['mean_dist_km'] = fire_daily['mean_dist_km'] / fire_daily['fire_count']
    except Exception as e:
        print(f"Skipping {f.name} due to: {e}")

fire_daily['date'] = pd.to_datetime(fire_daily['date']).dt.normalize()

# 3) Aggregate wind to daily mean and std across Punjab stations
wind_cols = ['Latitude','Longitude','Data Acquisition Time','Telemetry Hourly Wind Speed (Km/Hr)']
wind_file = DATA_DIR / 'wind_speed_tel_hr_punjab_sw_pb_1970_2025.csv'

wind_iter = pd.read_csv(wind_file, usecols=['Latitude','Longitude','Data Acquisition Time','Telemetry Hourly Wind Speed (Km/Hr)'], parse_dates=['Data Acquisition Time'], chunksize=200_000)
wind_aggs = []
for c in wind_iter:
    c = c[(c['Latitude'].between(lat_min, lat_max)) & (c['Longitude'].between(lon_min, lon_max))]
    if c.empty:
        continue
    c['date'] = pd.to_datetime(c['Data Acquisition Time']).dt.normalize()
    c = c.rename(columns={'Telemetry Hourly Wind Speed (Km/Hr)':'wind_kmh'})
    g = c.groupby('date').agg(mean_wind_kmh=('wind_kmh','mean'), std_wind_kmh=('wind_kmh','std'))
    wind_aggs.append(g)
if wind_aggs:
    wind_daily = pd.concat(wind_aggs).groupby(level=0).agg({'mean_wind_kmh':'mean','std_wind_kmh':'mean'}).reset_index()
    wind_daily['date'] = pd.to_datetime(wind_daily['date'])
else:
    wind_daily = pd.DataFrame(columns=['date','mean_wind_kmh','std_wind_kmh'])

# 4) Merge all: aqi_daily left join fire and wind on date, also add Delhi mean wind direction
merged = aqi_daily.merge(fire_daily, on='date', how='left').merge(wind_daily, on='date', how='left')
merged = merged.merge(delhi_daily_wind_dir, on='date', how='left')

# Fill missing fire data with zeros
for ccol in ['fire_count','total_frp','mean_frp','max_frp','weighted_frp','mean_dist_km','min_dist_km','mean_bearing_deg']:
    if ccol in merged.columns:
        merged[ccol] = merged[ccol].fillna(0)
# For missing wind, fill with global mean wind
merged['mean_wind_kmh'] = merged['mean_wind_kmh'].fillna(merged['mean_wind_kmh'].mean())
merged['std_wind_kmh'] = merged['std_wind_kmh'].fillna(merged['std_wind_kmh'].mean())
# If wind direction missing, fill with nan (we'll keep cos/sin)
merged['mean_wind_dir_deg'] = merged['mean_wind_dir_deg']
merged['mean_wind_dir_cos'] = merged['mean_wind_dir_cos']
merged['mean_wind_dir_sin'] = merged['mean_wind_dir_sin']

# 5) Add simple lag features (1-day lag for fire and wind)
merged = merged.sort_values('date')
merged['total_frp_lag1'] = merged['total_frp'].shift(1).fillna(0)
merged['fire_count_lag1'] = merged['fire_count'].shift(1).fillna(0)
merged['mean_wind_lag1'] = merged['mean_wind_kmh'].shift(1).fillna(merged['mean_wind_kmh'].mean())
merged['weighted_frp_lag1'] = merged['weighted_frp'].shift(1).fillna(0)

# 6) Compute alignment between fire bearing and Delhi mean wind direction (higher => winds blow from fires toward Delhi)
if 'mean_bearing_deg' in merged.columns and 'mean_wind_dir_deg' in merged.columns:
    # angular difference (wind_dir - bearing_to_delhi) -> 0 if wind comes from the bearing_to_delhi
    def ang_diff(wd, bd):
        if pd.isnull(wd) or pd.isnull(bd):
            return np.nan
        d = ((wd - bd + 180) % 360) - 180
        return d
    merged['fire_wind_ang_diff'] = merged.apply(lambda r: ang_diff(r.get('mean_wind_dir_deg', np.nan), r.get('mean_bearing_deg', np.nan)), axis=1)
    merged['fire_wind_alignment'] = np.cos(np.deg2rad(merged['fire_wind_ang_diff']))  # 1 -> aligned, -1 -> opposite
    # scale to 0-1
    merged['fire_wind_alignment01'] = (merged['fire_wind_alignment'] + 1.0) / 2.0
    # aligned weighted FRP
    merged['weighted_frp_aligned'] = merged['weighted_frp'] * merged['fire_wind_alignment01']
else:
    merged['fire_wind_ang_diff'] = np.nan
    merged['fire_wind_alignment'] = 0
    merged['fire_wind_alignment01'] = 0
    merged['weighted_frp_aligned'] = 0

# Save
merged.to_csv(OUT_CSV, index=False)
print(f"Saved merged dataset to {OUT_CSV} with {len(merged)} rows")

Saved merged dataset to a:\Software Projects\Delhi-AQI-Model\data\merged_aqi_fire_wind.csv with 1461 rows


C:\Users\Atharva Taras\AppData\Local\Temp\ipykernel_7244\294792690.py:70: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  for c in wind_iter:


In [6]:
# Additional processing: compute distance-weighted FRP and Delhi daily mean wind direction, then merge into the existing merged dataset
import pandas as pd
import numpy as np
from math import radians, sin, cos, asin, sqrt, atan2, degrees
from pathlib import Path

DATA_DIR = Path(r"a:/Software Projects/Delhi-AQI-Model/data")
MERGED_CSV = DATA_DIR / 'merged_aqi_fire_wind.csv'

# helpers

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return 6371.0 * 2 * asin(sqrt(a))

def bearing_deg(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = sin(dlon) * cos(lat2)
    y = cos(lat1)*sin(lat2) - sin(lat1)*cos(lat2)*cos(dlon)
    br = (degrees(atan2(x, y)) + 360) % 360
    return br

# Delhi centroid
DELHI_LAT, DELHI_LON = 28.7041, 77.1025

# Re-aggregate fire files with distance weighting
fire_files = [DATA_DIR / 'punjab_fires_2024.csv', DATA_DIR / 'punjab_fire_2020-2024.csv', DATA_DIR / 'fire_archive_J1V-C2_696273.csv']
lat_min, lat_max = 28.0, 33.5
lon_min, lon_max = 73.0, 77.5
agg_frames = []
for f in fire_files:
    try:
        it = pd.read_csv(f, parse_dates=['acq_date'], chunksize=200_000)
    except Exception as e:
        print(f"Skipping {f.name} read: {e}")
        continue
    for c in it:
        if not {'latitude','longitude','frp'}.issubset(c.columns):
            continue
        c = c[(c['latitude'].between(lat_min, lat_max)) & (c['longitude'].between(lon_min, lon_max))]
        if c.empty:
            continue
        c['date'] = pd.to_datetime(c['acq_date']).dt.normalize()
        # distances and bearing
        c['dist_km'] = c.apply(lambda r: haversine_km(r['latitude'], r['longitude'], DELHI_LAT, DELHI_LON), axis=1)
        c['bearing_to_delhi'] = c.apply(lambda r: bearing_deg(r['latitude'], r['longitude'], DELHI_LAT, DELHI_LON), axis=1)
        # weighting
        c['frp_weighted'] = c['frp'] / (c['dist_km'] + 1.0)
        c['b_x'] = np.cos(np.deg2rad(c['bearing_to_delhi'])) * c['frp']
        c['b_y'] = np.sin(np.deg2rad(c['bearing_to_delhi'])) * c['frp']
        g = c.groupby('date').agg(
            fire_count_fw=('frp','count'),
            total_frp_fw=('frp','sum'),
            weighted_frp_fw=('frp_weighted','sum'),
            dist_sum_fw=('dist_km','sum'),
            min_dist_km=('dist_km','min'),
            b_x_sum=('b_x','sum'),
            b_y_sum=('b_y','sum')
        )
        agg_frames.append(g)

if agg_frames:
    fire_fw = pd.concat(agg_frames).groupby(level=0).sum().reset_index().rename(columns={'index':'date'})
    fire_fw['date'] = pd.to_datetime(fire_fw['date']).dt.normalize()
    fire_fw['mean_dist_km'] = fire_fw['dist_sum_fw'] / fire_fw['fire_count_fw']
    fire_fw['mean_bearing_deg'] = (np.rad2deg(np.arctan2(fire_fw['b_y_sum'], fire_fw['b_x_sum'])) + 360) % 360
    fire_fw = fire_fw[['date','fire_count_fw','total_frp_fw','weighted_frp_fw','mean_dist_km','min_dist_km','mean_bearing_deg']]
else:
    fire_fw = pd.DataFrame(columns=['date','fire_count_fw','total_frp_fw','weighted_frp_fw','mean_dist_km','min_dist_km','mean_bearing_deg'])

# Delhi daily wind direction
try:
    ddf = pd.read_csv(DATA_DIR / 'delhi_air_quality_2024.csv', parse_dates=['event_timestamp'], usecols=['event_timestamp','wind_direction'])
    ddf = ddf.dropna(subset=['wind_direction'])
    ddf['date'] = ddf['event_timestamp'].dt.normalize()
    ddf['wind_direction'] = pd.to_numeric(ddf['wind_direction'], errors='coerce')
    ddf['wd_rad'] = np.deg2rad(ddf['wind_direction'])
    grp = ddf.groupby('date').agg(wd_cos_mean=('wd_rad', lambda s: np.cos(s).mean()), wd_sin_mean=('wd_rad', lambda s: np.sin(s).mean())).reset_index()
    grp['mean_wind_dir_rad'] = np.arctan2(grp['wd_sin_mean'], grp['wd_cos_mean'])
    grp['mean_wind_dir_deg'] = (np.rad2deg(grp['mean_wind_dir_rad']) + 360) % 360
    grp = grp.rename(columns={'wd_cos_mean':'mean_wind_dir_cos','wd_sin_mean':'mean_wind_dir_sin'})
    delhi_wind_daily = grp[['date','mean_wind_dir_deg','mean_wind_dir_cos','mean_wind_dir_sin']]
except Exception as e:
    print('Delhi wind direction aggregation failed:', e)
    delhi_wind_daily = pd.DataFrame(columns=['date','mean_wind_dir_deg','mean_wind_dir_cos','mean_wind_dir_sin'])

# Merge into existing merged dataset
if MERGED_CSV.exists():
    merged = pd.read_csv(MERGED_CSV, parse_dates=['date']).sort_values('date')
else:
    raise FileNotFoundError('Merged file not found. Run data processing cell first to create merged_aqi_fire_wind.csv')

merged = merged.merge(fire_fw, on='date', how='left')
merged = merged.merge(delhi_wind_daily, on='date', how='left')

# fillna defaults
for c in ['fire_count_fw','total_frp_fw','weighted_frp_fw','mean_dist_km','min_dist_km','mean_bearing_deg']:
    if c in merged.columns:
        merged[c] = merged[c].fillna(0)
merged['weighted_frp_lag1'] = merged['weighted_frp_fw'].shift(1).fillna(0)

# compute alignment between wind direction and mean bearing of fires
if ('mean_bearing_deg' in merged.columns) and ('mean_wind_dir_deg' in merged.columns):
    def ang_diff(wd, bd):
        if pd.isnull(wd) or pd.isnull(bd):
            return np.nan
        d = ((wd - bd + 180) % 360) - 180
        return d
    merged['fire_wind_ang_diff'] = merged.apply(lambda r: ang_diff(r.get('mean_wind_dir_deg', np.nan), r.get('mean_bearing_deg', np.nan)), axis=1)
    merged['fire_wind_alignment'] = np.cos(np.deg2rad(merged['fire_wind_ang_diff']))
    merged['fire_wind_alignment01'] = (merged['fire_wind_alignment'] + 1.0) / 2.0
    merged['weighted_frp_aligned'] = merged['weighted_frp_fw'] * merged['fire_wind_alignment01']
else:
    merged['fire_wind_ang_diff'] = np.nan
    merged['fire_wind_alignment'] = 0
    merged['fire_wind_alignment01'] = 0
    merged['weighted_frp_aligned'] = 0

# Save updated merged
merged.to_csv(MERGED_CSV, index=False)
print('Updated merged dataset with distance-weighted FRP & wind direction features and saved to', MERGED_CSV)

# quick sanity check
print(merged[['date','weighted_frp_fw','weighted_frp_aligned','mean_wind_dir_deg','mean_bearing_deg']].head())

Updated merged dataset with distance-weighted FRP & wind direction features and saved to a:\Software Projects\Delhi-AQI-Model\data\merged_aqi_fire_wind.csv
        date  weighted_frp_fw  weighted_frp_aligned  mean_wind_dir_deg  \
0 2021-01-01         0.518421                   NaN                NaN   
1 2021-01-02         0.039040                   NaN                NaN   
2 2021-01-03         0.123494                   NaN                NaN   
3 2021-01-04         0.477521                   NaN                NaN   
4 2021-01-05         0.000000                   NaN                NaN   

   mean_bearing_deg  
0        166.391332  
1        317.179167  
2        148.424738  
3        176.618923  
4          0.000000  


In [7]:
# Modeling: baseline models to predict daily AQI from fire and wind features (including new wind dir & weighted FRP)
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

DATA_DIR = Path(r"a:/Software Projects/Delhi-AQI-Model/data")
OUT_DIR = Path(r"a:/Software Projects/Delhi-AQI-Model/results")
OUT_DIR.mkdir(exist_ok=True)

merged = pd.read_csv(DATA_DIR / 'merged_aqi_fire_wind.csv', parse_dates=['date'])
merged = merged.sort_values('date').dropna(subset=['AQI'])

# Features
features = [
    'PM2.5','PM10','NO2','SO2','CO','Ozone',
    'fire_count','total_frp','mean_frp','max_frp',
    # distance-weighted and alignment features
    'weighted_frp_fw','weighted_frp_aligned','mean_dist_km','min_dist_km','fire_wind_alignment01',
    # wind direction representation
    'mean_wind_kmh','std_wind_kmh','mean_wind_dir_cos','mean_wind_dir_sin',
    # lags
    'total_frp_lag1','fire_count_lag1','mean_wind_lag1','weighted_frp_lag1'
]
# Ensure features exist
features = [f for f in features if f in merged.columns]
X = merged[features].fillna(0)
y = merged['AQI']

# Time-based split: train until 2023-12-31, test after (if data exists), otherwise 80/20 split
if merged['date'].max().year >= 2024 and merged['date'].min().year <= 2023:
    train_mask = merged['date'] < pd.to_datetime('2024-01-01')
    X_train, X_test = X[train_mask], X[~train_mask]
    y_train, y_test = y[train_mask], y[~train_mask]
else:
    n_train = int(0.8 * len(merged))
    X_train, X_test = X.iloc[:n_train], X.iloc[n_train:]
    y_train, y_test = y.iloc[:n_train], y.iloc[n_train:]

# Train models
lr = LinearRegression()
lr.fit(X_train, y_train)
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Predict and evaluate
models = {'LinearRegression': lr, 'RandomForest': rf}
metrics = []
for name, m in models.items():
    y_pred = m.predict(X_test)
    # Compute RMSE as sqrt of MSE to avoid compatibility issues with sklearn versions
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    metrics.append({'model':name, 'rmse':rmse, 'mae':mae})

metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(OUT_DIR / 'aqi_model_metrics.csv', index=False)
print(metrics_df)

# Feature importances from RF
fi = pd.DataFrame({'feature':features, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)
fi.to_csv(OUT_DIR / 'feature_importances.csv', index=False)

# Save a short markdown summary (avoid tabulate dependency by using to_string())
with open(OUT_DIR / 'aqi_model_metrics.md', 'w') as fh:
    fh.write('# AQI Model metrics\n\n')
    fh.write(metrics_df.to_string(index=False))
    fh.write('\n\n## Top features from RandomForest\n')
    fh.write(fi.head(10).to_string(index=False))

print('Saved metrics and feature importances to results/')

              model       rmse        mae
0  LinearRegression  51.152341  30.147833
1      RandomForest  30.908094  20.998962
Saved metrics and feature importances to results/


In [8]:
# EXPERIMENTS: feature reduction, cleaning/normalization, model variety, and time-series forecasting (include new features)
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error

DATA_DIR = Path(r"a:/Software Projects/Delhi-AQI-Model/data")
OUT_DIR = Path(r"a:/Software Projects/Delhi-AQI-Model/results")
OUT_DIR.mkdir(exist_ok=True)

merged = pd.read_csv(DATA_DIR / 'merged_aqi_fire_wind.csv', parse_dates=['date']).sort_values('date').reset_index(drop=True)
# keep data between 2021-01-01 and 2024-12-31 to limit range
merged = merged[(merged['date']>=pd.to_datetime('2021-01-01')) & (merged['date']<=pd.to_datetime('2024-12-31'))].copy()
merged = merged.fillna(0)

# Baseline metrics storage
experiment_results = []

# Helper functions
def eval_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    return {'rmse':rmse,'mae':mae,'y_test':y_test, 'y_pred':y_pred}

# A plotting helper to save predicted vs actual and residuals
def plot_preds(y_test, y_pred, title, fname):
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    sns.scatterplot(x=y_test, y=y_pred, alpha=0.6)
    plt.plot([y_test.min(), y_test.max()],[y_test.min(), y_test.max()], 'r--')
    plt.xlabel('Actual AQI'); plt.ylabel('Predicted AQI'); plt.title(title + ' (pred vs actual)')
    plt.subplot(1,2,2)
    res = y_test - y_pred
    sns.histplot(res, kde=True)
    plt.title('Residuals')
    plt.tight_layout()
    plt.savefig(OUT_DIR / fname)
    plt.close()

# Time-based split helper
def time_split(df, split_date='2024-01-01'):
    train = df[df['date'] < pd.to_datetime(split_date)]
    test = df[df['date'] >= pd.to_datetime(split_date)]
    return train, test

# Keep copies for experiments
base = merged.copy()

# --------------------
# Experiment 1: Feature reduction
# --------------------
exp_name = 'feature_reduction'
# select a compact set of features (domain-driven)
features_small = ['PM2.5','PM10','total_frp_lag1','fire_count_lag1','mean_wind_lag1','weighted_frp_lag1','fire_wind_alignment01']
features_small = [f for f in features_small if f in base.columns]
train, test = time_split(base)
X_train, X_test = train[features_small].fillna(0), test[features_small].fillna(0)
y_train, y_test = train['AQI'], test['AQI']

models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
}
for name, m in models.items():
    r = eval_model(m, X_train, y_train, X_test, y_test)
    experiment_results.append({'experiment':exp_name,'model':name,'rmse':r['rmse'],'mae':r['mae']})
    plot_preds(r['y_test'], r['y_pred'], f'{exp_name} - {name}', f'{exp_name}_{name}_preds.png')

# --------------------
# Experiment 2: Cleaning & normalization
# --------------------
exp_name = 'cleaning_normalization'
df = base.copy()
# Outlier removal on PM2.5 and PM10 using IQR
for col in ['PM2.5','PM10']:
    if col in df.columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        mask = ~((df[col] < (q1 - 3*iqr)) | (df[col] > (q3 + 3*iqr)))
        df = df[mask]
df = df.reset_index(drop=True)

# Impute missing with median and scale with RobustScaler
features_all = [c for c in ['PM2.5','PM10','NO2','SO2','CO','Ozone','fire_count','total_frp','mean_frp','max_frp','weighted_frp_fw','weighted_frp_aligned','mean_dist_km','min_dist_km','mean_wind_kmh','std_wind_kmh','mean_wind_dir_cos','mean_wind_dir_sin','total_frp_lag1','fire_count_lag1','mean_wind_lag1','weighted_frp_lag1','fire_wind_alignment01'] if c in df.columns]
X = df[features_all].fillna(df[features_all].median())
scaler = RobustScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features_all, index=X.index)

train = df[df['date'] < pd.to_datetime('2024-01-01')]
test = df[df['date'] >= pd.to_datetime('2024-01-01')]
X_train = X_scaled.loc[train.index]
X_test = X_scaled.loc[test.index]
y_train = train['AQI']; y_test = test['AQI']

models = {
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42, max_iter=5000),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=200, random_state=42)
}
for name, m in models.items():
    r = eval_model(m, X_train, y_train, X_test, y_test)
    experiment_results.append({'experiment':exp_name,'model':name,'rmse':r['rmse'],'mae':r['mae']})
    plot_preds(r['y_test'], r['y_pred'], f'{exp_name} - {name}', f'{exp_name}_{name}_preds.png')

# --------------------
# Experiment 3: Model exploration with simple tuning
# --------------------
exp_name = 'model_exploration'
train = base[base['date'] < pd.to_datetime('2024-01-01')]
test = base[base['date'] >= pd.to_datetime('2024-01-01')]
features = [c for c in ['PM2.5','PM10','NO2','SO2','CO','Ozone','fire_count','total_frp','mean_frp','max_frp','weighted_frp_fw','weighted_frp_aligned','mean_dist_km','min_dist_km','mean_wind_kmh','std_wind_kmh','mean_wind_dir_cos','mean_wind_dir_sin','total_frp_lag1','fire_count_lag1','mean_wind_lag1','weighted_frp_lag1','fire_wind_alignment01'] if c in base.columns]
X_train = train[features].fillna(0); X_test = test[features].fillna(0)
y_train, y_test = train['AQI'], test['AQI']

# try SVR (with small sample due to compute), HistGradientBoosting, and tuned RandomForest
svr = SVR(C=1.0, kernel='rbf')
hgb = HistGradientBoostingRegressor(max_iter=200, random_state=42)
param_grid = {'max_depth':[5,10,20], 'n_estimators':[100]}
rf = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), param_grid, cv=3, scoring='neg_mean_squared_error')

for name, m in [('SVR',svr), ('HistGB',hgb), ('RF_tuned', rf)]:
    m.fit(X_train, y_train)
    if name == 'RF_tuned':
        best = m.best_estimator_
        y_pred = best.predict(X_test)
    else:
        y_pred = m.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    experiment_results.append({'experiment':exp_name,'model':name,'rmse':rmse,'mae':mae})
    plot_preds(y_test, y_pred, f'{exp_name} - {name}', f'{exp_name}_{name}_preds.png')

# --------------------
# Experiment 4: Time series forecasting (autoregressive with exogenous variables)
# --------------------
exp_name = 'time_series_ar'
# create lag features of AQI
ts = base.copy()
for lag in range(1,8):
    ts[f'AQI_lag_{lag}'] = ts['AQI'].shift(lag)
# use last 7 lags + exogenous (fire, wind, weighted frp and alignment)
exog = [f'AQI_lag_{i}' for i in range(1,8)] + [c for c in ['total_frp_lag1','mean_wind_lag1','fire_count_lag1','weighted_frp_lag1','fire_wind_alignment01'] if c in ts.columns]
# drop NA from lagging
ts = ts.dropna(subset=exog + ['AQI']).reset_index(drop=True)
train = ts[ts['date'] < pd.to_datetime('2024-01-01')]
test = ts[ts['date'] >= pd.to_datetime('2024-01-01')]
X_train = train[exog]; X_test = test[exog]; y_train = train['AQI']; y_test = test['AQI']

# use a robust regressor (GradientBoosting) as autoregressive forecaster
ar_model = GradientBoostingRegressor(n_estimators=300, random_state=42)
ar_model.fit(X_train, y_train)
y_pred = ar_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred)); mae = mean_absolute_error(y_test, y_pred)
experiment_results.append({'experiment':exp_name,'model':'GB_AR','rmse':rmse,'mae':mae})
plot_preds(y_test, y_pred, f'{exp_name} - GB_AR', f'{exp_name}_GB_AR_preds.png')

# Also produce a rolling-origin forecast to simulate real forecasting (one-step ahead iteratively)
roll_preds = []
roll_index = []
history = train.copy()
for idx,row in test.iterrows():
    Xrow = row[exog].values.reshape(1,-1)
    p = ar_model.predict(Xrow)[0]
    roll_preds.append(p)
    roll_index.append(row['date'])
# metrics
roll_rmse = np.sqrt(mean_squared_error(test['AQI'], roll_preds)); roll_mae = mean_absolute_error(test['AQI'], roll_preds)
experiment_results.append({'experiment':exp_name,'model':'GB_AR_roll','rmse':roll_rmse,'mae':roll_mae})
# plot rolling preds
plt.figure(figsize=(10,4))
plt.plot(test['date'], test['AQI'], label='actual')
plt.plot(test['date'], roll_preds, label='rolling_pred')
plt.xticks(rotation=30)
plt.legend(); plt.title('Time series rolling forecast')
plt.tight_layout(); plt.savefig(OUT_DIR / f'{exp_name}_rolling_forecast.png'); plt.close()

# --------------------
# Aggregate results and save
# --------------------
res_df = pd.DataFrame(experiment_results)
res_df.to_csv(OUT_DIR / 'experiment_summary.csv', index=False)

# Plot summary RMSEs
plt.figure(figsize=(8,4))
sns.barplot(data=res_df, x='model', y='rmse', hue='experiment')
plt.xticks(rotation=45); plt.title('RMSE by model and experiment'); plt.tight_layout(); plt.savefig(OUT_DIR / 'experiment_rmse_summary.png'); plt.close()

plt.figure(figsize=(8,4))
sns.barplot(data=res_df, x='model', y='mae', hue='experiment')
plt.xticks(rotation=45); plt.title('MAE by model and experiment'); plt.tight_layout(); plt.savefig(OUT_DIR / 'experiment_mae_summary.png'); plt.close()

print('Experiments completed and saved to results/')

a:\Software Projects\Delhi-AQI-Model\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
a:\Software Projects\Delhi-AQI-Model\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
a:\Software Projects\Delhi-AQI-Model\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
a:\Software Projects\Delhi-AQI-Model\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
a:\Software Projects\Delhi-AQI-Model\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X do

Experiments completed and saved to results/


<!-- Summary of experiments and key plots -->
# 📋 Experiment Summary — AQI prediction (Delhi)

**Quick summary:**
- **Best model so far:** `RF_tuned` (RandomForest) — **RMSE ≈ 29.94** (best observed) ✅
- Feature-reduced RandomForest and cleaned GradientBoosting/ElasticNet performed similarly (RMSE ~31).
- Autoregressive time-series (GB on lags + exogenous) did **not** outperform the supervised models.

**Optuna tuning (new):**
- **HistGradientBoosting (Optuna)** — **RMSE 30.57**, **MAE 21.78** 🔧
- **LightGBM (Optuna)** — **RMSE 34.14**, **MAE 24.72** ⚠️
- Interpretation: Optuna-tuned HistGB improved the HistGB baseline but did **not** beat the best `RF_tuned` result; LightGBM underperformed in these trials.

**Artifacts saved (results/):**
- `experiment_summary.csv`, `experiment_rmse_summary.png`, `experiment_mae_summary.png`
- `aqi_model_metrics.csv`, `feature_importances.csv`
- Hyperparameter search: `hyperopt_results.csv`, `hyperopt_rmse_comparison.png`, `best_rf_randomized.pkl`, `best_gb_randomized.pkl`
- Optuna outputs: `optuna_results.csv`, `optuna_rmse_comparison.png`, `optuna_best_HistGB_optuna_preds.png`, `optuna_best_LightGBM_optuna_preds.png`, `best_hist_optuna.pkl`, `best_lgb_optuna.pkl`

**How to reproduce quickly:**
1. Run the Experiments cell (the cell that begins with "EXPERIMENTS: feature reduction...").
2. Re-run the Hyperparameter Search cell ("HYPERPARAMETER SEARCH") and the Optuna cell ("OPTUNA TUNING: HistGradientBoosting and LightGBM") to reproduce the searches and plots.

**Interpretation & next steps 💡:**
- The tuned RandomForest remains the **most promising** model. The Optuna HistGradientBoosting results are close but not better.
- Recommended next step: run a **focused Optuna/Bayesian search around the best RandomForest region** (n_estimators, max_depth, max_features, min_samples_leaf) to try and squeeze further RMSE gains (this is computationally heavier but targeted).
- Other options: export the notebook as an HTML report for sharing, or try additional transport-aware feature engineering (e.g., fire arrival time windows, sector-weighted FRP).

---

### Key summary plots

**RMSE comparison:**

![](results/experiment_rmse_summary.png)

**Optuna RMSE comparison:**

![](results/optuna_rmse_comparison.png)

**Best model — RF_tuned predictions (pred vs actual + residuals):**

![](results/model_exploration_RF_tuned_preds.png)

**Optuna best model — predictions (pred vs actual + residuals):**

![](results/optuna_best_HistGB_optuna_preds.png)

**Time-series rolling forecast (for reference):**

![](results/time_series_ar_rolling_forecast.png)

> Note: All plots and CSVs are in the `results/` folder. If you'd like, I can run a focused RF Optuna search next or export this notebook as a shareable HTML report.


In [9]:
# HYPERPARAMETER SEARCH (time-aware RandomizedSearchCV for RF and GradientBoosting)
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import randint, uniform

DATA_DIR = Path(r"a:/Software Projects/Delhi-AQI-Model/data")
OUT_DIR = Path(r"a:/Software Projects/Delhi-AQI-Model/results")
OUT_DIR.mkdir(exist_ok=True)

merged = pd.read_csv(DATA_DIR / 'merged_aqi_fire_wind.csv', parse_dates=['date']).sort_values('date').reset_index(drop=True)
merged = merged[(merged['date']>=pd.to_datetime('2021-01-01')) & (merged['date']<=pd.to_datetime('2024-12-31'))].copy()
merged = merged.fillna(0)

# Features as used in the tuned experiments
features = [c for c in ['PM2.5','PM10','NO2','SO2','CO','Ozone','weighted_frp_fw','weighted_frp_aligned','mean_dist_km','min_dist_km','mean_wind_kmh','std_wind_kmh','mean_wind_dir_cos','mean_wind_dir_sin','total_frp_lag1','fire_count_lag1','mean_wind_lag1','weighted_frp_lag1','fire_wind_alignment01'] if c in merged.columns]

# Train/test split by date
train = merged[merged['date'] < pd.to_datetime('2024-01-01')]
test = merged[merged['date'] >= pd.to_datetime('2024-01-01')]
X_train = train[features]; X_test = test[features]
y_train = train['AQI']; y_test = test['AQI']

# TimeSeriesSplit for cross-validation (preserve ordering)
tscv = TimeSeriesSplit(n_splits=5)

# RandomForest param distribution
rf = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_params = {
    'n_estimators': randint(100, 800),
    'max_depth': randint(3, 40),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 6),
    'max_features': ['auto','sqrt','log2', 0.5, 0.7]
}

# GradientBoosting param distribution (sklearn GBT)
gb = GradientBoostingRegressor(random_state=42)
gb_params = {
    'n_estimators': randint(100, 1000),
    'learning_rate': uniform(0.01, 0.3),
    'max_depth': randint(2, 8),
    'subsample': uniform(0.5, 0.5),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 6)
}

# Search settings
n_iter_rf = 50
n_iter_gb = 40
scoring = 'neg_mean_squared_error'

print('Starting RandomizedSearchCV for RandomForest...')
rf_search = RandomizedSearchCV(rf, rf_params, n_iter=n_iter_rf, cv=tscv, scoring=scoring, random_state=42, n_jobs=-1, verbose=2)
rf_search.fit(X_train, y_train)
print('RF search done. Best params:', rf_search.best_params_)

print('Starting RandomizedSearchCV for GradientBoosting...')
gb_search = RandomizedSearchCV(gb, gb_params, n_iter=n_iter_gb, cv=tscv, scoring=scoring, random_state=42, n_jobs=-1, verbose=2)
gb_search.fit(X_train, y_train)
print('GB search done. Best params:', gb_search.best_params_)

# Evaluate on test set
best_rf = rf_search.best_estimator_
y_pred_rf = best_rf.predict(X_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_mae = mean_absolute_error(y_test, y_pred_rf)

best_gb = gb_search.best_estimator_
y_pred_gb = best_gb.predict(X_test)
gb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_gb))
gb_mae = mean_absolute_error(y_test, y_pred_gb)

# Save results
res = pd.DataFrame([{
    'model':'RF_randomized','rmse':rf_rmse,'mae':rf_mae,'best_params':str(rf_search.best_params_)
},{
    'model':'GB_randomized','rmse':gb_rmse,'mae':gb_mae,'best_params':str(gb_search.best_params_)
}])
res.to_csv(OUT_DIR / 'hyperopt_results.csv', index=False)

# Save pickles of best models
joblib.dump(best_rf, OUT_DIR / 'best_rf_randomized.pkl')
joblib.dump(best_gb, OUT_DIR / 'best_gb_randomized.pkl')

# Plot comparison and predictions
plt.figure(figsize=(8,4))
plt.bar(res['model'], res['rmse'], color=['C0','C1'])
plt.title('RMSE on test set for randomized search best models')
plt.ylabel('RMSE'); plt.tight_layout(); plt.savefig(OUT_DIR / 'hyperopt_rmse_comparison.png'); plt.close()

# Predictions vs actual for best model (choose the best rmse)
best_name = res.loc[res['rmse'].idxmin(),'model']
if 'RF' in best_name:
    best_pred = y_pred_rf
    model_label = 'RandomForest_randomized'
else:
    best_pred = y_pred_gb
    model_label = 'GradientBoosting_randomized'

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
sns.scatterplot(x=y_test, y=best_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()],[y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual AQI'); plt.ylabel('Predicted AQI'); plt.title(f'{model_label} (pred vs actual)')
plt.subplot(1,2,2)
resid = y_test - best_pred
sns.histplot(resid, kde=True)
plt.title('Residuals')
plt.tight_layout(); plt.savefig(OUT_DIR / f'hyperopt_best_{model_label}_preds.png'); plt.close()

print('Hyperparameter search finished. Results saved to results/hyperopt_results.csv and plots.')

Starting RandomizedSearchCV for RandomForest...
Fitting 5 folds for each of 50 candidates, totalling 250 fits


a:\Software Projects\Delhi-AQI-Model\.venv\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
45 fits failed out of a total of 250.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
25 fits failed with the following error:
Traceback (most recent call last):
  File "a:\Software Projects\Delhi-AQI-Model\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "a:\Software Projects\Delhi-AQI-Model\.venv\Lib\site-packages\sklearn\base.py", line 1329, in wrapper
    estimator._validate_params()
  File "a:\Software Projects\Delhi-AQI-Model\.venv\Lib\site-packages\sklearn\base.py", line 492, in _validate_params
    validate_p

RF search done. Best params: {'max_depth': 5, 'max_features': 0.7, 'min_samples_leaf': 3, 'min_samples_split': 8, 'n_estimators': 120}
Starting RandomizedSearchCV for GradientBoosting...
Fitting 5 folds for each of 40 candidates, totalling 200 fits
GB search done. Best params: {'learning_rate': np.float64(0.014789875666064259), 'max_depth': 3, 'min_samples_leaf': 4, 'min_samples_split': 5, 'n_estimators': 466, 'subsample': np.float64(0.8416317594127292)}
Hyperparameter search finished. Results saved to results/hyperopt_results.csv and plots.


In [2]:
# OPTUNA TUNING: HistGradientBoosting and LightGBM (time-series aware)
import sys
import subprocess
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error

OUT_DIR = Path(r"a:/Software Projects/Delhi-AQI-Model/results")
OUT_DIR.mkdir(exist_ok=True)
DATA_DIR = Path(r"a:/Software Projects/Delhi-AQI-Model/data")

# ensure optuna (and lightgbm) available
try:
    import optuna
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna"], stdout=subprocess.DEVNULL)
    import optuna

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "lightgbm"], stdout=subprocess.DEVNULL)
        import lightgbm as lgb
        HAS_LGB = True
    except Exception:
        print('LightGBM not available; skipping LightGBM tuning.')

merged = pd.read_csv(DATA_DIR / 'merged_aqi_fire_wind.csv', parse_dates=['date']).sort_values('date').reset_index(drop=True)
merged = merged[(merged['date']>=pd.to_datetime('2021-01-01')) & (merged['date']<=pd.to_datetime('2024-12-31'))].copy()
merged = merged.fillna(0)

features = [c for c in ['PM2.5','PM10','NO2','SO2','CO','Ozone','weighted_frp_fw','weighted_frp_aligned','mean_dist_km','min_dist_km','mean_wind_kmh','std_wind_kmh','mean_wind_dir_cos','mean_wind_dir_sin','total_frp_lag1','fire_count_lag1','mean_wind_lag1','weighted_frp_lag1','fire_wind_alignment01'] if c in merged.columns]
train = merged[merged['date'] < pd.to_datetime('2024-01-01')]
test = merged[merged['date'] >= pd.to_datetime('2024-01-01')]
X_train = train[features].values; X_test = test[features].values
y_train = train['AQI'].values; y_test = test['AQI'].values

tscv = TimeSeriesSplit(n_splits=5)

# ---------------------
# HistGradientBoosting objective
# ---------------------

def hist_objective(trial):
    params = {
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.3),
        'max_iter': trial.suggest_int('max_iter', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 2, 16),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'l2_regularization': trial.suggest_loguniform('l2_regularization', 1e-6, 10.0),
        'max_bins': trial.suggest_int('max_bins', 64, 255)
    }
    cv_scores = []
    for train_idx, val_idx in tscv.split(X_train):
        Xtr, Xval = X_train[train_idx], X_train[val_idx]
        ytr, yval = y_train[train_idx], y_train[val_idx]
        model = HistGradientBoostingRegressor(random_state=42, **params)
        model.fit(Xtr, ytr)
        pred = model.predict(Xval)
        cv_scores.append(np.sqrt(mean_squared_error(yval, pred)))
    return np.mean(cv_scores)

# run Optuna for HistGB
hist_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
hist_study.optimize(hist_objective, n_trials=40)
print('HistGB best params:', hist_study.best_params)

# Evaluate best HistGB on test
best_hist = HistGradientBoostingRegressor(random_state=42, **hist_study.best_params)
best_hist.fit(X_train, y_train)
hist_pred = best_hist.predict(X_test)
hist_rmse = np.sqrt(mean_squared_error(y_test, hist_pred))
hist_mae = mean_absolute_error(y_test, hist_pred)

# Save Hist results
hist_res = {'model':'HistGB_optuna','rmse':hist_rmse,'mae':hist_mae,'best_params':str(hist_study.best_params)}

# Save model
joblib.dump(best_hist, OUT_DIR / 'best_hist_optuna.pkl')

# ---------------------
# LightGBM objective (if available)
# ---------------------
lgb_res = None
if HAS_LGB:
    def lgb_objective(trial):
        param = {
            'objective':'regression', 'boosting_type':'gbdt', 'verbosity':-1, 'random_state':42,
            'num_leaves': trial.suggest_int('num_leaves', 16, 256),
            'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.3),
            'n_estimators': trial.suggest_int('n_estimators', 100, 2000),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
            'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
            'subsample_freq': trial.suggest_int('subsample_freq', 0, 10),
            'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
            'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-8, 10.0),
            'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-8, 10.0)
        }
        cv_scores = []
        for train_idx, val_idx in tscv.split(X_train):
            Xtr, Xval = X_train[train_idx], X_train[val_idx]
            ytr, yval = y_train[train_idx], y_train[val_idx]
            model = lgb.LGBMRegressor(**param)
            # try early stopping if supported; otherwise, do a plain fit
            try:
                model.fit(Xtr, ytr, eval_set=[(Xval, yval)], early_stopping_rounds=50, verbose=False)
            except TypeError:
                model.fit(Xtr, ytr)
            pred = model.predict(Xval)
            cv_scores.append(np.sqrt(mean_squared_error(yval, pred)))
        return np.mean(cv_scores)

    lgb_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    lgb_study.optimize(lgb_objective, n_trials=50)
    print('LightGBM best params:', lgb_study.best_params)

    # Evaluate best LGB on test
    best_lgb = lgb.LGBMRegressor(**lgb_study.best_params)
    try:
        best_lgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], early_stopping_rounds=50, verbose=False)
    except TypeError:
        best_lgb.fit(X_train, y_train)
    lgb_pred = best_lgb.predict(X_test)
    lgb_rmse = np.sqrt(mean_squared_error(y_test, lgb_pred))
    lgb_mae = mean_absolute_error(y_test, lgb_pred)
    lgb_res = {'model':'LightGBM_optuna','rmse':lgb_rmse,'mae':lgb_mae,'best_params':str(lgb_study.best_params)}
    joblib.dump(best_lgb, OUT_DIR / 'best_lgb_optuna.pkl')

# ---------------------
# Save and plot results
# ---------------------
all_res = [hist_res]
if lgb_res:
    all_res.append(lgb_res)
res_df = pd.DataFrame(all_res)
res_df.to_csv(OUT_DIR / 'optuna_results.csv', index=False)

# plot comparison
plt.figure(figsize=(6,3))
plt.bar(res_df['model'], res_df['rmse'].astype(float), color=['C0','C1'][:len(res_df)])
plt.title('Optuna-tuned models RMSE')
plt.ylabel('RMSE'); plt.tight_layout(); plt.savefig(OUT_DIR / 'optuna_rmse_comparison.png'); plt.close()

# best model preds plot
best_idx = res_df['rmse'].astype(float).idxmin()
best_model_name = res_df.loc[best_idx,'model']
if best_model_name == 'HistGB_optuna':
    best_preds = hist_pred
else:
    best_preds = lgb_pred

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
sns.scatterplot(x=y_test, y=best_preds, alpha=0.6)
plt.plot([y_test.min(), y_test.max()],[y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual AQI'); plt.ylabel('Predicted AQI'); plt.title(f'{best_model_name} (pred vs actual)')
plt.subplot(1,2,2)
resid = y_test - best_preds
sns.histplot(resid, kde=True)
plt.title('Residuals')
plt.tight_layout(); plt.savefig(OUT_DIR / f'optuna_best_{best_model_name}_preds.png'); plt.close()

print('Optuna tuning complete. Results saved to results/optuna_results.csv and plots.')

[I 2026-01-10 20:17:26,401] A new study created in memory with name: no-name-93fd71a8-450e-47b1-8790-c84a3ec907a5
C:\Users\Atharva Taras\AppData\Local\Temp\ipykernel_25176\4069720753.py:55: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.3),
C:\Users\Atharva Taras\AppData\Local\Temp\ipykernel_25176\4069720753.py:59: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'l2_regularization': trial.suggest_loguniform('l2_regularization', 1e-6, 10.0),
[I 2026-01-10 20:17:32,254] Trial 0 finished with value: 38.03378789235657 and parameters: {'learning_rate': 0.008468008575248327, 'max_iter': 956, 'max_depth': 12, 'min

HistGB best params: {'learning_rate': 0.015736507932657535, 'max_iter': 537, 'max_depth': 3, 'min_samples_leaf': 13, 'l2_regularization': 0.09600018296639787, 'max_bins': 150}


[I 2026-01-10 20:19:04,372] A new study created in memory with name: no-name-d0ef9fb6-5cb8-4710-9735-66b561b86b85
C:\Users\Atharva Taras\AppData\Local\Temp\ipykernel_25176\4069720753.py:99: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.3),
C:\Users\Atharva Taras\AppData\Local\Temp\ipykernel_25176\4069720753.py:102: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
C:\Users\Atharva Taras\AppData\Local\Temp\ipykernel_25176\4069720753.py:104: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optun

LightGBM best params: {'num_leaves': 17, 'learning_rate': 0.015088691406128611, 'n_estimators': 1626, 'min_child_samples': 5, 'subsample': 0.7330949718714018, 'subsample_freq': 2, 'colsample_bytree': 0.8316538322481921, 'reg_alpha': 1.0684774614427359e-08, 'reg_lambda': 0.02653585372372143}


a:\Software Projects\Delhi-AQI-Model\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Optuna tuning complete. Results saved to results/optuna_results.csv and plots.


## Focused Optuna Tuning — RandomForest (`RF_tuned`)

This cell runs a focused Optuna study for `RandomForestRegressor` optimizing RMSE with a time-series-aware CV (TimeSeriesSplit). It saves trial results to `results/optuna_rf_trials.csv` and best params to `results/rf_best_params.json`.

Notes: Run this cell interactively; it may take time depending on `n_trials` and dataset size.

In [2]:
# Focused Optuna for RandomForest (RF_tuned)
import sys, subprocess, json
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import joblib

OUT_DIR = Path(r"a:/Software Projects/Delhi-AQI-Model/results")
OUT_DIR.mkdir(exist_ok=True)
DATA_DIR = Path(r"a:/Software Projects/Delhi-AQI-Model/data")

try:
    import optuna
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna"], stdout=subprocess.DEVNULL)
    import optuna

# Load merged dataset (same features as used previously)
merged = pd.read_csv(DATA_DIR / 'merged_aqi_fire_wind.csv', parse_dates=['date']).sort_values('date').reset_index(drop=True)
merged = merged[(merged['date']>=pd.to_datetime('2021-01-01')) & (merged['date']<=pd.to_datetime('2024-12-31'))].copy()
merged = merged.fillna(0)

features = [c for c in ['PM2.5','PM10','NO2','SO2','CO','Ozone','weighted_frp_fw','weighted_frp_aligned','mean_dist_km','min_dist_km','mean_wind_kmh','std_wind_kmh','mean_wind_dir_cos','mean_wind_dir_sin','total_frp_lag1','fire_count_lag1','mean_wind_lag1','weighted_frp_lag1','fire_wind_alignment01'] if c in merged.columns]
train = merged[merged['date'] < pd.to_datetime('2024-01-01')]
test = merged[merged['date'] >= pd.to_datetime('2024-01-01')]
X_train = train[features].values; X_test = test[features].values
y_train = train['AQI'].values; y_test = test['AQI'].values

tscv = TimeSeriesSplit(n_splits=5)

def rf_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 40),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 12),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 6),
        'max_features': trial.suggest_categorical('max_features', ['sqrt','log2', 0.5, 0.7]),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False])
    }
    cv_scores = []
    for train_idx, val_idx in tscv.split(X_train):
        Xtr, Xval = X_train[train_idx], X_train[val_idx]
        ytr, yval = y_train[train_idx], y_train[val_idx]
        model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
        model.fit(Xtr, ytr)
        pred = model.predict(Xval)
        cv_scores.append(np.sqrt(mean_squared_error(yval, pred)))
    return np.mean(cv_scores)

# Create study and optimize
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
N_TRIALS = 60
study.optimize(rf_objective, n_trials=N_TRIALS, show_progress_bar=True)

print('RF Optuna best params:', study.best_params, 'best_value (RMSE):', study.best_value)

# Train best model on full training set and evaluate on test set
best_rf = RandomForestRegressor(random_state=42, n_jobs=-1, **study.best_params)
best_rf.fit(X_train, y_train)
pred_test = best_rf.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, pred_test))
print('Test RMSE for best RF:', test_rmse)

# Save study trials and best params and model
trials_df = study.trials_dataframe(attrs=('number','value','params','state'))
trials_df.to_csv(OUT_DIR / 'optuna_rf_trials.csv', index=False)
with open(OUT_DIR / 'rf_best_params.json','w') as fh: json.dump(study.best_params, fh, indent=2)
joblib.dump(best_rf, OUT_DIR / 'best_rf_optuna.pkl')

print('Saved trials to results/optuna_rf_trials.csv and model to results/best_rf_optuna.pkl')

[I 2026-01-18 13:46:19,835] A new study created in memory with name: no-name-fe66e213-29b8-41df-922e-afde6aaac2e4


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-01-18 13:46:22,455] Trial 0 finished with value: 37.78271900096667 and parameters: {'n_estimators': 437, 'max_depth': 39, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 0.7, 'bootstrap': False}. Best is trial 0 with value: 37.78271900096667.
[I 2026-01-18 13:46:23,778] Trial 1 finished with value: 36.250132224634186 and parameters: {'n_estimators': 118, 'max_depth': 39, 'min_samples_split': 11, 'min_samples_leaf': 2, 'max_features': 0.7, 'bootstrap': True}. Best is trial 1 with value: 36.250132224634186.
[I 2026-01-18 13:46:29,486] Trial 2 finished with value: 43.13421075464958 and parameters: {'n_estimators': 651, 'max_depth': 8, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': 'log2', 'bootstrap': True}. Best is trial 1 with value: 36.250132224634186.
[I 2026-01-18 13:46:35,368] Trial 3 finished with value: 44.88461727552808 and parameters: {'n_estimators': 647, 'max_depth': 9, 'min_samples_split': 2, 'min_samples_leaf': 6, 'max_features': 'sqrt